# Final Inferential Validation — Registered-host experiment

**Scope:** targeted statistical triangulation of approved Sprint 1 and EDA 1, **not** a new EDA or final causal recommendation. All technical documentation is in English. The experimental comparison is among **14,708 registered hosts**, not verified full intention-to-treat. Source CSVs are immutable.

**Primary source timestamps:** `assignments.completed_at` sets baseline `[2025-07-18,2025-08-18)` and experiment `[2025-08-18,2025-11-20)` in UTC; earliest non-null `payment_logs.captured_at` identifies a recorded capture by `completed_at +24h`. Pre-completion captures count as already observed at D1. `payment_logs.created_at` is **not** a failure or capture timestamp.

**Design safeguards:** host-level allocation/resampling; group counts preserved; no project-independent inference; D1 project-weighted rate conditional on completion (potential treatment selection). The recorded exposure file does not confirm the exact original randomization procedure, assignment time or complete randomized population.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
while not (ROOT/'src'/'experiment_impact.py').exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
if not (ROOT/'src'/'experiment_impact.py').exists():
    raise FileNotFoundError('Run this notebook inside the Turno project root (with src/ intact).')
sys.path.insert(0,str(ROOT))
from src.experiment_impact import (build_project_panel,build_host_panel,
    bootstrap_host_differences,SEED)
from src.inferential_validation import (paired_period_host_panel,paired_d1_bootstrap,
    hc3_adjusted_host_metrics,host_label_permutation,holm_adjust,influence_by_host)

hosts=pd.read_csv(ROOT/'data/raw/experiment_exposures.csv',dtype='string')
assignments=pd.read_csv(ROOT/'data/raw/assignments.csv',dtype='string')
logs=pd.read_csv(ROOT/'data/raw/payment_logs.csv',dtype='string')
project=build_project_panel(hosts,assignments,logs)
panel=build_host_panel(hosts,project)
paired=paired_period_host_panel(hosts,project)
assert len(hosts)==len(panel)==len(paired)==14708
assert len(project)==len(assignments)==279847 and len(logs)==306957
assert paired.experiment_projects.sum()==int(project.in_experiment.sum())==201302
assert paired.baseline_projects.sum()==int(project.baseline.sum())
assert panel.completed.sum()==paired.experiment_projects.sum()
assert not panel.host_id.duplicated().any() and not paired.host_id.duplicated().any()
print('Source rows / registered hosts / baseline completions / experiment completions:',
      len(logs),len(hosts),int(project.baseline.sum()),int(project.in_experiment.sum()))
print('Registered group sizes:',panel.experiment_group.value_counts().to_dict())

Source rows / registered hosts / baseline completions / experiment completions: 306957 14708 78545 201302
Registered group sizes: {'control': 7377, 'test': 7331}


## A. Baseline D1 and the change in group contrasts

D1 is the **project-weighted** share of completed projects with at least one first observed capture timestamp at or before completion + 24 elapsed hours. This is a historical descriptive contrast between **later-registered** Control and Test labels, not evidence of a treatment effect before treatment. Bootstrap (2,000 draws; seed 20261029) re-samples registered **hosts within group**, retaining each host's baseline and experimental numerator and denominator in the same draw. The third contrast is experimental difference minus baseline difference; it is **not** automatically a causal difference-in-differences estimator. Neither parallel trends nor the original allocation mechanism are verified.

In [2]:
d1,period_counts=paired_d1_bootstrap(paired,n_boot=2000,seed=SEED)
assert d1.valid_replicates.eq(2000).all()
assert sum(c['baseline_projects'] for c in period_counts.values())==int(project.baseline.sum())
assert sum(c['experiment_projects'] for c in period_counts.values())==len(project.loc[project.in_experiment])
show=d1.copy()
for field in ['control','test','difference','ci_low','ci_high']:
    show[field]=show[field]*100
print('D1 rates and contrasts in percentage points (except group rates, which are %):')
display(show.round(4))
print('Group host, project, capture and contributing-host denominators:')
display(pd.DataFrame.from_dict(period_counts,orient='index'))
print('Historical baseline vs experiment contrast change:',round(100*d1.iloc[2]['difference'],4),'pp')

D1 rates and contrasts in percentage points (except group rates, which are %):


,contrast,control,test,difference,ci_low,ci_high,valid_replicates
0,D1 baseline,98.9414,98.3825,-0.5589,-1.5081,0.1923,2000
1,D1 experiment,98.8336,98.1034,-0.7302,-1.1220,-0.3322,2000
2,D1 change in contrasts (exploratory),NaN,NaN,-0.1713,-0.7130,0.5838,2000


Group host, project, capture and contributing-host denominators:


,hosts,baseline_projects,experiment_projects,baseline_captured,experiment_captured,hosts_with_baseline,hosts_with_experiment
control,7377,38731,98855,38321,97702,5200,6938
test,7331,39814,102447,39170,100504,5084,6934


Historical baseline vs experiment contrast change: -0.1713 pp


## B. Baseline-only adjustment, registered-label permutation and Holm family

Host-level linear adjustment uses Test indicator and **pre-period completed-project count** only, with HC3 robust 95% intervals. Zero-activity hosts remain in the sample. M1, M2 and M4 use the same covariate; **no D1 regression is forced**, because D1 conditions on completed projects, changes project weights, and may be affected by treatment-dependent project selection. Historical D1 is directly assessed in A instead.

The alternative **4,000-permutation, two-sided random-label test** permutes *whole host labels*, maintaining the observed group sizes (7,331 Test / 7,377 Control), and recomputes D1 as a joint numerator/denominator project ratio for each group. This **assumes exchangeability among registered labels**. It is **not exact randomization inference** because undocumented assignment strata and selection into the register cannot be recovered. The Monte Carlo p-value uses the plus-one correction; its lower resolution is `1/4001`. Holm controls familywise error for this small **post hoc** four-test family and does not confer pre-registration.

In [3]:
adjusted=hc3_adjusted_host_metrics(panel)
boot=bootstrap_host_differences(panel,n_boot=2000,seed=SEED)
permutation,_=host_label_permutation(panel,paired,n_perm=4000,seed=SEED)
permutation['p_holm']=holm_adjust(permutation.p_permutation_two_sided.to_numpy())
metric_names={'M1':'M1_completed_per_host','M2':'M2_first_ever_per_host','M4':'M4_captured_D14_per_host'}
rows=[]
for metric in ['M1','M2','M4','D1']:
    if metric=='D1':
        base=d1.loc[d1.contrast.eq('D1 experiment')].iloc[0]
        row={'metric':metric,'raw_diff':base.difference,'raw_ci_low':base.ci_low,'raw_ci_high':base.ci_high,
             'control':base.control,'test':base.test,'unit':'project capture proportion'}
    else:
        base=boot.loc[boot.metric.eq(metric_names[metric])].iloc[0]
        row={'metric':metric,'raw_diff':base.difference,'raw_ci_low':base.ci95_low,'raw_ci_high':base.ci95_high,
             'control':base.control,'test':base.test,'unit':'projects per registered host'}
        adj=adjusted.loc[adjusted.metric.eq(metric)].iloc[0]
        row.update({'adjusted_diff':adj.adjusted_difference,'adjusted_ci_low':adj.adjusted_ci_low,'adjusted_ci_high':adj.adjusted_ci_high})
    perm=permutation.loc[permutation.metric.eq(metric)].iloc[0]
    assert np.isclose(row['raw_diff'],perm.observed_difference,atol=1e-12)
    row.update({'p_perm':perm.p_permutation_two_sided,'p_holm':perm.p_holm})
    rows.append(row)
evidence=pd.DataFrame(rows)
assert len(evidence)==4 and not evidence[['raw_diff','raw_ci_low','raw_ci_high','p_perm','p_holm']].isna().any().any()
print('Estimates are proportions for D1 and projects/host for M1/M2/M4; see units. HC3 is not used for D1.')
display(evidence.round(6))
print('Permutation details: 4,000 host-label permutations, seed',SEED,'; family selected after initial results.')
print('Adjustment design: Test indicator + preperiod project completions (host-grain OLS, HC3).')

Estimates are proportions for D1 and projects/host for M1/M2/M4; see units. HC3 is not used for D1.


,metric,raw_diff,raw_ci_low,raw_ci_high,control,test,unit,adjusted_diff,adjusted_ci_low,adjusted_ci_high,p_perm,p_holm
0,M1,0.574058,-0.332201,1.598279,13.400434,13.974492,projects per registered host,0.174202,-0.204823,0.553227,0.254186,0.508373
1,M2,0.014767,0.003345,0.025504,0.136099,0.150866,projects per registered host,0.015638,0.004495,0.026781,0.011247,0.033742
2,M4,0.557368,-0.342614,1.562049,13.327504,13.884872,projects per registered host,0.160374,-0.219774,0.540521,0.264934,0.508373
3,D1,-0.007302,-0.011220,-0.003322,0.988336,0.981034,project capture proportion,NaN,NaN,NaN,0.000250,0.001000


Permutation details: 4,000 host-label permutations, seed 20261029 ; family selected after initial results.
Adjustment design: Test indicator + preperiod project completions (host-grain OLS, HC3).


## C. Host-activity concentration and influence

All registered hosts, including zero-completion hosts, contribute to M1. Describe the distribution per group (median, p75, p90, p95, p99, max), the share of total project completions contributed by the **top 1% of hosts within each group**, and one **predeclared uniform diagnostic**: cap each registered host's *observed* experiment completions at the **pooled 99th percentile**, applying the **same** numeric cap to both groups. The uncapped result remains primary. The cap is based on a post-treatment variable and **does not** define a comparable causal subgroup or a new primary effect estimate.

In [4]:
concentration,influence=influence_by_host(panel,quantile=.99)
assert concentration.projects.sum()==int(project.in_experiment.sum())
assert int(concentration.hosts.sum())==len(hosts)
assert concentration.pooled_p99_cap.nunique()==1
print('Host-level activity distribution and top 1% contribution:')
display(concentration.round(4))
print('Influence sensitivity (Test − Control projects / registered host):')
display(influence.round(6))
assert np.isclose(influence.iloc[0].test_minus_control,evidence.loc[evidence.metric.eq('M1'),'raw_diff'].item())

Host-level activity distribution and top 1% contribution:


,group,hosts,projects,mean,median,p75,p90,p95,p99,max,top_1pct_n,top_1pct_share,pooled_p99_cap,capped_mean,share_hosts_zero
0,control,7377,98855,13.4004,7.0,16.0,28.0,42.0,112.24,548,74,0.1429,112.0,12.6092,0.0595
1,test,7331,102447,13.9745,7.0,16.0,28.0,43.0,111.70,1311,74,0.1801,112.0,12.5889,0.0542


Influence sensitivity (Test − Control projects / registered host):


,diagnostic,test_minus_control,pooled_cap
0,Full sample mean difference,0.574058,NaN
1,Same pooled-p99 capped project count (post-tre...,-0.020322,112.0


## D. Audit and interpretation limits

**Test interpretation:** An interval including zero is **not evidence of equivalence/noninferiority**; no margins were defined. M2 is a source-flagged first historical completion, **not** verified new-host acquisition. M4 is observed timestamp-based D14 capture **not** settled funds. D1 conditions on completed projects, whose existence may depend on treatment. Baseline D1 similarity or difference does **not** prove balance or trends. Uniform-label permutation rests on a hypothetical exchangeability mechanism not recovered from the source. Holm is retrospective; all EDA 1 RMSTs, additional horizons, subgroups and sensitivities remain exploratory and outside the four-test family.

**Decision evidence to carry forward:** (1) baseline D1 contrast vs experimental D1 and descriptive contrast change; (2) M2 adjusted + multiplicity; (3) M1/M4 baseline adjustment; (4) influence of extreme hosts; (5) whether permutation signs agree with prior bootstrap. No Ship/Kill decision, full-settlement claim, or additional model is made.

In [5]:
# Data-grain, paired-bootstrap, permutation and multiplicity gates.
assert evidence.metric.tolist()==['M1','M2','M4','D1']
assert int((panel.completed==0).sum())==int((paired.experiment_projects==0).sum())
assert (paired.experiment_captured<=paired.experiment_projects).all()
assert (paired.baseline_captured<=paired.baseline_projects).all()
assert int(panel.captured_D14.sum())<=int(panel.completed.sum())
assert len(permutation)==4 and (permutation.p_holm>=permutation.p_permutation_two_sided).all()
assert np.isclose(float(evidence.loc[evidence.metric.eq('D1'),'raw_diff'].item()),
                  float(d1.loc[d1.contrast.eq('D1 experiment'),'difference'].item()))
print('PASS: host census, zero-host preservation, period/grain reconciliation, paired D1,')
print('      fixed-seed host permutation, HC3 adjustment, Holm and influence diagnostic.')
print('NOTE: Original randomization mechanism not documented; p-values are conditional exploratory checks.')

PASS: host census, zero-host preservation, period/grain reconciliation, paired D1,
      fixed-seed host permutation, HC3 adjustment, Holm and influence diagnostic.
NOTE: Original randomization mechanism not documented; p-values are conditional exploratory checks.


### Independent source reconciliation — baseline and experiment D1

This focused check uses Python standard-library `csv.DictReader` and `datetime` rather than Pandas transformations to independently rebuild the earliest populated capture timestamp and the D1 project capture rate for each group and period. No source records are changed. It verifies the new baseline contrast, its denominators and the experiment contrast before interpreting them.

In [6]:
import csv
from datetime import datetime, timedelta
from collections import defaultdict

audit_groups={}
with (ROOT/'data/raw/experiment_exposures.csv').open(newline='') as f:
    for rec in csv.DictReader(f):
        audit_groups[rec['host_id']]=rec['experiment_group']
assert len(audit_groups)==len(hosts)
first_time={}
with (ROOT/'data/raw/payment_logs.csv').open(newline='') as f:
    for rec in csv.DictReader(f):
        stamp=rec['captured_at']
        if stamp:
            tt=datetime.fromisoformat(stamp)
            assignment=rec['assignment_id']
            if assignment not in first_time or tt<first_time[assignment]:
                first_time[assignment]=tt
source_counts=defaultdict(lambda:[0,0])
with (ROOT/'data/raw/assignments.csv').open(newline='') as f:
    for rec in csv.DictReader(f):
        moment=datetime.fromisoformat(rec['completed_at'])
        period=('baseline' if datetime(2025,7,18)<=moment<datetime(2025,8,18)
                else 'experiment' if datetime(2025,8,18)<=moment<datetime(2025,11,20)
                else None)
        if period is None:
            continue
        group=audit_groups[rec['host_id']]
        numer=source_counts[(period,group)]
        numer[1]+=1
        first=first_time.get(rec['assignment_id'])
        if first is not None and first<=moment+timedelta(hours=24):
            numer[0]+=1
for period in ('baseline','experiment'):
    for group in ('control','test'):
        counted=source_counts[(period,group)]
        expected=period_counts[group]
        assert counted==[expected[period+'_captured'],expected[period+'_projects']]
assert sum(x[1] for x in source_counts.values())==int(project.baseline.sum()+project.in_experiment.sum())
print('PASS: independent csv/datetime audit agrees with group-specific D1 numerators and denominators:')
print(dict(source_counts))

PASS: independent csv/datetime audit agrees with group-specific D1 numerators and denominators:
{('experiment', 'control'): [97702, 98855], ('experiment', 'test'): [100504, 102447], ('baseline', 'control'): [38321, 38731], ('baseline', 'test'): [39170, 39814]}
